# LLM-Guided Optimization Demo

This notebook demonstrates how to use LLMs to automatically discover and apply performance optimizations.

We'll:
1. Define a realistic function with performance issues
2. Use the LLM-guided optimizer to get optimization suggestions
3. Safely execute the optimized code
4. Verify correctness and measure performance improvements

In [ ]:
# Add parent directory to path to import modules
import sys
sys.path.insert(0, '..')

from optimizers.optimizer import optimize_with_llm
from optimizers.llm_client import create_llm_client
from verifiers.verifier import check_correctness, compare_performance, print_performance_comparison
import os

## 1. Configure LLM Client

You can use either:
- **DummyLLMClient**: For offline testing (no API key needed)
- **OpenAILLMClient**: For real optimizations (requires OPENAI_API_KEY)

Set `USE_REAL_LLM = True` and configure your API key to use real LLM.

In [ ]:
# Configuration
USE_REAL_LLM = False  # Set to True to use OpenAI (requires API key)

if USE_REAL_LLM:
    # Check if API key is set
    api_key = os.getenv('OPENAI_API_KEY')
    if not api_key:
        print("⚠️  OPENAI_API_KEY not set. Using dummy client instead.")
        llm_client = create_llm_client('dummy')
    else:
        llm_client = create_llm_client('openai', model='gpt-3.5-turbo')
        print(f"✅ Using OpenAI model: {llm_client.get_model_name()}")
else:
    llm_client = create_llm_client('dummy')
    print(f"ℹ️  Using {llm_client.get_model_name()} (offline mode)")

## 2. Define a Function to Optimize

Let's create a function that processes data inefficiently.

In [ ]:
def process_data(numbers):
    """Process a list of numbers with several inefficiencies."""
    # Inefficiency 1: Multiple passes over the data
    positive = [x for x in numbers if x > 0]
    squared = [x**2 for x in positive]
    
    # Inefficiency 2: Unnecessary intermediate list
    total = sum([x for x in squared])
    
    # Inefficiency 3: Repeated lookups
    result = []
    for x in squared:
        if x < total / len(squared):
            result.append(x)
    
    return result

# Test the function
test_input = list(range(-50, 51))
print(f"Result for test input: {process_data(test_input)[:5]}...")

## 3. Run LLM-Guided Optimization

Now let's use the optimizer to profile and optimize this function.

In [ ]:
# Run optimization
print("🔍 Profiling and optimizing...\n")

optimization_result = optimize_with_llm(
    process_data,
    test_input,
    llm_client=llm_client,
    n_runs=10
)

print("✅ Optimization complete!\n")
print("="*60)
print(f"Model used: {optimization_result.model_used}")
print("="*60)

## 4. Review the Optimization

Let's examine what the LLM suggested.

In [ ]:
print("\n📝 EXPLANATION:")
print("-" * 60)
print(optimization_result.explanation)

print("\n\n💻 ORIGINAL CODE:")
print("-" * 60)
print(optimization_result.original_code)

print("\n\n✨ OPTIMIZED CODE:")
print("-" * 60)
print(optimization_result.optimized_code)

## 5. Execute and Test the Optimized Code

⚠️ **Safety Warning**: Using `exec()` to run LLM-generated code can be dangerous in production.
This is acceptable for research/demo purposes with controlled inputs.

In production, you would:
1. Review the code manually
2. Use sandboxing/containerization
3. Apply security scanning

In [ ]:
# Create a namespace for the optimized function
namespace = {}

try:
    # Execute the optimized code
    exec(optimization_result.optimized_code, namespace)
    
    # Extract the optimized function
    # The function name should be the same as the original
    optimized_func = namespace.get('process_data') or namespace.get('process_data_optimized')
    
    if optimized_func is None:
        # Try to find any function in the namespace
        funcs = [v for v in namespace.values() if callable(v) and not v.__name__.startswith('_')]
        if funcs:
            optimized_func = funcs[0]
        else:
            raise ValueError("No optimized function found in namespace")
    
    print(f"✅ Successfully loaded optimized function: {optimized_func.__name__}")
    
except Exception as e:
    print(f"❌ Error executing optimized code: {e}")
    print("\nThis can happen with the dummy client. Try using a real LLM.")
    optimized_func = None

## 6. Verify Correctness

Before measuring performance, let's verify the optimized function produces correct results.

In [ ]:
if optimized_func:
    # Define test cases
    test_cases = [
        ((list(range(-50, 51)),), {}),
        ((list(range(-10, 11)),), {}),
        (([1, 2, 3, 4, 5],), {}),
        (([-5, -4, -3, -2, -1],), {}),
        (([],), {}),
    ]
    
    # Check correctness
    passed, errors = check_correctness(process_data, optimized_func, test_cases)
    
    if passed:
        print("✅ All correctness tests PASSED!")
        print(f"   Tested {len(test_cases)} test cases")
    else:
        print("❌ Some correctness tests FAILED:")
        for error in errors:
            print(f"\n{error}")
else:
    print("⚠️  Skipping correctness check (no optimized function available)")

## 7. Measure Performance Improvement

Finally, let's measure the performance improvement.

In [ ]:
if optimized_func:
    # Create larger test input for performance measurement
    perf_test_input = list(range(-5000, 5001))
    
    # Compare performance
    comparison = compare_performance(
        process_data,
        optimized_func,
        perf_test_input,
        n_runs=50
    )
    
    print_performance_comparison(comparison)
    
    # Interpret results
    if comparison['speedup'] > 1.1:
        print("🎉 Significant performance improvement achieved!")
    elif comparison['speedup'] > 1.0:
        print("✅ Modest performance improvement achieved.")
    else:
        print("⚠️  No performance improvement (or regression).")
        print("    This can happen with the dummy client or if the LLM didn't find good optimizations.")
else:
    print("⚠️  Skipping performance comparison (no optimized function available)")

## Summary

This notebook demonstrated the complete LLM-guided optimization pipeline:

1. ✅ **Profiling**: Identified performance bottlenecks
2. ✅ **LLM Optimization**: Generated optimized code
3. ✅ **Correctness Verification**: Ensured behavior is preserved
4. ✅ **Performance Measurement**: Quantified improvements

### Notes

- The **dummy client** returns placeholder code for testing the pipeline
- For **real optimizations**, set `USE_REAL_LLM = True` and provide an OpenAI API key
- Always **review LLM-generated code** before using it in production
- Consider **sandboxing** when executing AI-generated code

### Next Steps

Try this workflow with your own functions! Consider:
- Functions with nested loops
- Data processing pipelines
- Numerical computations
- String manipulations